In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torchvision import datasets, transforms
from torch.utils.data import DataLoader, WeightedRandomSampler
import matplotlib.pyplot as plt
import numpy as np
import os

torch.set_num_threads(8)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

BASE_DIR  = r"C:\Users\Rushikesh\OneDrive\CODES\SelfHealingNN"
os.chdir(BASE_DIR)

IMAGE_SIZE = 224
BATCH_SIZE = 32

# ✅ Enhanced augmentations, no Normalize — [0, 1] range for VAE sigmoid
train_transforms = transforms.Compose([
    transforms.Grayscale(num_output_channels=1),
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.RandomAffine(degrees=0, translate=(0.1, 0.1), scale=(0.9, 1.1)),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor(),
])

train_dataset = datasets.ImageFolder(
    root=os.path.join(BASE_DIR, "medical_data", "train"),
    transform=train_transforms
)

# ⚖️ Balanced sampling
targets = np.array(train_dataset.targets)
class_counts = np.bincount(targets)
sample_weights = (1.0 / class_counts)[targets]
sampler = WeightedRandomSampler(sample_weights, len(sample_weights))

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, sampler=sampler, num_workers=0, pin_memory=True)

def add_medical_noise(image_tensor, noise_factor=0.3):
    noise = torch.randn_like(image_tensor) * noise_factor
    return torch.clamp(image_tensor + noise, 0., 1.)

print(f"🖥️ Device: {device}")
if device.type == 'cuda':
    print(f"🎮 GPU: {torch.cuda.get_device_name(0)}")
print(f"✅ Train: {len(train_dataset)} images")
print(f"✅ Classes: {train_dataset.classes}")
print(f"⚖️  Distribution: {dict(zip(train_dataset.classes, class_counts))}")

✅ Train: 5216 images
✅ Classes: ['NORMAL', 'PNEUMONIA']
⚖️  Distribution: {'NORMAL': np.int64(1341), 'PNEUMONIA': np.int64(3875)}


In [ ]:
# ✅ Deeper VAE for 224x224 — 5 encoder/decoder blocks, latent_dim=256
class DeepMedicalVAE(nn.Module):
    def __init__(self, latent_dim=256):
        super(DeepMedicalVAE, self).__init__()
        self.latent_dim = latent_dim

        # Encoder: 224→112→56→28→14→7
        self.encoder = nn.Sequential(
            nn.Conv2d(1,  32,  3, stride=2, padding=1), nn.BatchNorm2d(32),  nn.ReLU(),
            nn.Conv2d(32, 64,  3, stride=2, padding=1), nn.BatchNorm2d(64),  nn.ReLU(),
            nn.Conv2d(64, 128, 3, stride=2, padding=1), nn.BatchNorm2d(128), nn.ReLU(),
            nn.Conv2d(128,256, 3, stride=2, padding=1), nn.BatchNorm2d(256), nn.ReLU(),
            nn.Conv2d(256,512, 3, stride=2, padding=1), nn.BatchNorm2d(512), nn.ReLU(),
        )

        self.fc_mu     = nn.Linear(512 * 7 * 7, latent_dim)
        self.fc_logvar = nn.Linear(512 * 7 * 7, latent_dim)
        self.fc_decode = nn.Linear(latent_dim, 512 * 7 * 7)

        # Decoder: 7→14→28→56→112→224
        self.decoder = nn.Sequential(
            nn.ConvTranspose2d(512, 256, 3, stride=2, padding=1, output_padding=1), nn.BatchNorm2d(256), nn.ReLU(),
            nn.ConvTranspose2d(256, 128, 3, stride=2, padding=1, output_padding=1), nn.BatchNorm2d(128), nn.ReLU(),
            nn.ConvTranspose2d(128, 64,  3, stride=2, padding=1, output_padding=1), nn.BatchNorm2d(64),  nn.ReLU(),
            nn.ConvTranspose2d(64,  32,  3, stride=2, padding=1, output_padding=1), nn.BatchNorm2d(32),  nn.ReLU(),
            nn.ConvTranspose2d(32,  1,   3, stride=2, padding=1, output_padding=1),
            nn.Sigmoid()
        )

    def reparameterize(self, mu, logvar):
        std = torch.exp(0.5 * logvar)
        return mu + torch.randn_like(std) * std

    def forward(self, x):
        h      = self.encoder(x).view(-1, 512 * 7 * 7)
        mu     = self.fc_mu(h)
        logvar = self.fc_logvar(h)
        z      = self.reparameterize(mu, logvar)
        h2     = F.relu(self.fc_decode(z)).view(-1, 512, 7, 7)
        return self.decoder(h2), mu, logvar

total = sum(p.numel() for p in DeepMedicalVAE().parameters())
print(f"✅ VAE ready (5 blocks for 224x224, latent_dim=256) | Parameters: {total:,}")

✅ VAE ready (with BatchNorm) | Parameters: 7,084,929


In [3]:
def vae_loss(recon_x, x, mu, logvar, beta=1.0):
    BCE = F.binary_cross_entropy(recon_x, x, reduction='sum')
    KLD = -0.5 * torch.sum(1 + logvar - mu.pow(2) - logvar.exp())
    return BCE + beta * KLD

print("✅ Loss function ready")

✅ Loss function ready


In [ ]:
os.makedirs(os.path.join(BASE_DIR, "models"), exist_ok=True)

medical_healer = DeepMedicalVAE().to(device)
optimizer = optim.Adam(medical_healer.parameters(), lr=1e-3)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, patience=3, factor=0.5)

EPOCHS = 30  # ✅ More epochs for convergence on GPU

print(f"🚀 Training VAE for {EPOCHS} epochs on {device}...")
print(f"   Batches per epoch : {len(train_loader)}\n")

for epoch in range(EPOCHS):
    medical_healer.train()
    total_loss = 0

    for batch_idx, (clean_images, _) in enumerate(train_loader):
        clean_images = clean_images.to(device)
        noisy_images = add_medical_noise(clean_images, 0.4)

        optimizer.zero_grad()
        recon_images, mu, logvar = medical_healer(noisy_images)
        loss = vae_loss(recon_images, clean_images, mu, logvar, beta=0.5)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(medical_healer.parameters(), 1.0)
        optimizer.step()

        total_loss += loss.item()

        if batch_idx % 50 == 0:
            avg = total_loss / (batch_idx + 1)
            print(f"  Epoch {epoch+1}/{EPOCHS} | Batch {batch_idx:>3}/{len(train_loader)} | Loss: {avg:,.1f}")

    avg_loss = total_loss / len(train_loader)
    scheduler.step(avg_loss)
    lr = optimizer.param_groups[0]['lr']
    print(f"✅ Epoch {epoch+1}/{EPOCHS} done | Avg Loss: {avg_loss:,.1f} | LR: {lr:.1e}\n")

torch.save(medical_healer.state_dict(), os.path.join(BASE_DIR, "models", "medical_healer.pth"))
print("💾 Saved → models/medical_healer.pth")

🚀 Training VAE for 20 epochs on CPU...
   Batches per epoch : 326

  Epoch 1/20 | Batch   0/326 | Loss: 219,852.0
  Epoch 1/20 | Batch  50/326 | Loss: 170,975.3
  Epoch 1/20 | Batch 100/326 | Loss: 164,260.2
  Epoch 1/20 | Batch 150/326 | Loss: 161,010.2
  Epoch 1/20 | Batch 200/326 | Loss: 158,701.2
  Epoch 1/20 | Batch 250/326 | Loss: 157,080.5
  Epoch 1/20 | Batch 300/326 | Loss: 155,859.7
✅ Epoch 1/20 done | Avg Loss: 155,341.7 | LR: 1.0e-03

  Epoch 2/20 | Batch   0/326 | Loss: 147,841.8
  Epoch 2/20 | Batch  50/326 | Loss: 148,811.9
  Epoch 2/20 | Batch 100/326 | Loss: 148,872.1
  Epoch 2/20 | Batch 150/326 | Loss: 148,638.1
  Epoch 2/20 | Batch 200/326 | Loss: 148,415.2
  Epoch 2/20 | Batch 250/326 | Loss: 148,267.5
  Epoch 2/20 | Batch 300/326 | Loss: 148,170.4
✅ Epoch 2/20 done | Avg Loss: 148,053.0 | LR: 1.0e-03

  Epoch 3/20 | Batch   0/326 | Loss: 148,952.7
  Epoch 3/20 | Batch  50/326 | Loss: 147,496.7
  Epoch 3/20 | Batch 100/326 | Loss: 147,546.4
  Epoch 3/20 | Batch 150

KeyboardInterrupt: 

In [ ]:
medical_healer.eval()
images, _ = next(iter(train_loader))
images = images.to(device)
noisy  = add_medical_noise(images, 0.4)

with torch.no_grad():
    healed, _, _ = medical_healer(noisy)

fig, axes = plt.subplots(3, 4, figsize=(16, 10))
titles = ["Original", "Corrupted", "Healed"]
rows   = [images.cpu(), noisy.cpu(), healed.cpu()]

for row_idx, (row_imgs, title) in enumerate(zip(rows, titles)):
    for col in range(4):
        axes[row_idx, col].imshow(row_imgs[col].squeeze().detach(), cmap='gray')
        axes[row_idx, col].set_title(title, color='green' if title == 'Healed' else 'black')
        axes[row_idx, col].axis('off')

plt.suptitle("VAE Healing Quality (224x224)", fontsize=14)
plt.tight_layout()
plt.show()